# 04 – Classical Baselines & Geometry (Colab)

Evaluates the two classical baselines that the U-Net must beat:
1. Perspective correction (OpenCV)
2. Background division + CLAHE illumination baseline


In [ ]:
!pip -q install opencv-python scikit-image albumentations jiwer pytesseract pyyaml
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Adjust for Colab vs local
COLAB = 'google.colab' in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRescue'
else:
    PROJECT_ROOT = '.'

sys.path.insert(0, PROJECT_ROOT)

from src.geometry import correct_perspective, detect_document_corners
from src.classical import background_division, clahe_enhance, adaptive_threshold, classical_pipeline
from src.metrics import compute_psnr, compute_ssim, compute_rmse

In [ ]:
# ── Demo: Perspective correction on own photos ─────────────────────────────
OWN_PHOTOS = os.path.join(PROJECT_ROOT, 'data', 'own_photos')
photos = sorted(Path(OWN_PHOTOS).glob('*.jpg'))[:4]

if photos:
    fig, axes = plt.subplots(2, len(photos), figsize=(14, 6))
    for col, p in enumerate(photos):
        orig = np.array(Image.open(p).convert('RGB'))
        corr, ok = correct_perspective(orig)
        axes[0, col].imshow(orig);  axes[0, col].set_title('Original');   axes[0, col].axis('off')
        axes[1, col].imshow(corr);  axes[1, col].set_title('Corrected' if ok else 'Failed'); axes[1, col].axis('off')
    plt.suptitle('Perspective correction (own photos)')
    plt.tight_layout(); plt.show()
else:
    print('No own photos found yet. Add them to data/own_photos/')

In [ ]:
# ── Demo: Classical illumination baseline on SD7K test ─────────────────────
SD7K_TEST_INPUT  = os.path.join(PROJECT_ROOT, 'data', 'sd7k_512', 'test', 'input')
SD7K_TEST_TARGET = os.path.join(PROJECT_ROOT, 'data', 'sd7k_512', 'test', 'target')

test_files = sorted(os.listdir(SD7K_TEST_INPUT))[:4]

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col, fn in enumerate(test_files):
    inp      = np.array(Image.open(os.path.join(SD7K_TEST_INPUT, fn)).convert('RGB'))
    tgt      = np.array(Image.open(os.path.join(SD7K_TEST_TARGET, fn)).convert('RGB'))
    baseline = classical_pipeline(inp)
    
    axes[0, col].imshow(inp);      axes[0, col].set_title('Input');    axes[0, col].axis('off')
    axes[1, col].imshow(baseline); axes[1, col].set_title('Baseline'); axes[1, col].axis('off')
    axes[2, col].imshow(tgt);      axes[2, col].set_title('Target');   axes[2, col].axis('off')

plt.suptitle('Classical baseline: background division + CLAHE')
plt.tight_layout(); plt.show()

In [ ]:
# ── Quantitative: PSNR/SSIM on full test set ───────────────────────────────
from tqdm.auto import tqdm

psnr_raw, ssim_raw = [], []
psnr_classic, ssim_classic = [], []

all_test = sorted(os.listdir(SD7K_TEST_INPUT))
for fn in tqdm(all_test, desc='Evaluating classical baseline'):
    inp  = np.array(Image.open(os.path.join(SD7K_TEST_INPUT, fn)).convert('RGB'))
    tgt  = np.array(Image.open(os.path.join(SD7K_TEST_TARGET, fn)).convert('RGB'))
    baseline = classical_pipeline(inp)
    
    psnr_raw.append(compute_psnr(inp, tgt))
    ssim_raw.append(compute_ssim(inp, tgt))
    psnr_classic.append(compute_psnr(baseline, tgt))
    ssim_classic.append(compute_ssim(baseline, tgt))

import pandas as pd
import numpy as np
results = pd.DataFrame({
    'Arm': ['Raw input', 'Classical baseline'],
    'PSNR (dB)': [np.mean(psnr_raw), np.mean(psnr_classic)],
    'SSIM':      [np.mean(ssim_raw), np.mean(ssim_classic)],
})
print(results.to_string(index=False))
os.makedirs(os.path.join(PROJECT_ROOT, 'results', 'tables'), exist_ok=True)
results.to_csv(os.path.join(PROJECT_ROOT, 'results', 'tables', 'classical_baseline.csv'), index=False)

In [ ]:
# ── Corner detection failure analysis ──────────────────────────────────────
# Check how often perspective detection fails on own photos
if photos:
    success, fail = 0, 0
    for p in sorted(Path(OWN_PHOTOS).glob('*.jpg')):
        img = np.array(Image.open(p).convert('RGB'))
        corners = detect_document_corners(img)
        if corners is not None:
            success += 1
        else:
            fail += 1
            print(f'  Detection failed: {p.name}')
    print(f'Corner detection: {success} success, {fail} failures out of {success+fail}')